# CNC Guard — 2. Fasea: Aurreprozesatzea eta Entrenamendua

**AABD Espezializazioa · 5071, 5072, 5073 moduluak**

Helburua: `ai4i2020_train.csv` datuetatik hutsegiteak iragarriko dituen eredu bat entrenatu eta `models/cnc_failure_model.pkl` gisa gorde.

## 1. Liburutegiak eta Karga

Beharrezko liburutegiak inportatu eta datuak kargatu.

## 2. EDA — Datu-azterketa
Aztertu: zutabe motak eta balioen barrutiak, datu-balioen banaketa, korrelazioak

## 3. Aurreprozesatzea
Prestatu datuak entrenamendurako, feature engineering...
## 4. Train / Validation Zatiketa
## 5. Entrenamendua

Eredu bat (gutxienez) entrenatu. Aukerakoa: hainbat eredu konparatu.
## 6. Ebaluazioa
Ebaluatu metrika desberdinekin.
## 7. Eredua Gorde

Scoring-api-k `/models/cnc_failure_model.pkl` bilatu behar du.

In [ ]:
# zure kodea hemen
# # Liburutegiak
from pathlib import Path

import math
import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (ConfusionMatrixDisplay, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split

# 1. datuabasea pasa entrenamenduako
df = pd.read_csv('../data/ai4i2020_train.csv')


# 2. Dimentsioa inprimatu (matrizea)
print(f"Dataset-ren dimentsioa: {df.shape[0]} filaka eta  {df.shape[1]} zutabeka")

# 3. null baloreak berrikusi (NaN / missing values)
nulos = df.isnull().sum()
print("\n--- Balio nulluak ---")
print(nulos[nulos > 0] if nulos.sum() > 0 else "Ez daude balio nulluak dataset-an.")


# 4. Fila duplikatuen konprobaketa
duplicados = df.duplicated().sum()
print(f"\nFila duplikatuak (kop): {duplicados}")

# 5. Datu motak eta filak ikusi
print("\n--- Datu motak ---")
print(df.dtypes)
df.head()

#DATUEN GARBIKETA

# 1. objetuaren bariablea definitu (Target)
target = "machine_failure"

# 2. Ezabatu beharreko zutabeak (data leakage) / erroreko datuak
# Si el modelo viera esto, sabría de antemano si falló y no aprendería de los sensores
leakage_cols = [
    "failure_TWF", "failure_HDF", "failure_PWF", "failure_OSF", "failure_RNF"
]

# 3. Columnas identificadoras que no generalizan
id_cols = ["machine_id"]

# 4. Zutabe erabilgarriak (erabiliko ditugun sensoreak )
feature_cols = [
    "machine_type",    # Categórica: L, M, H
    "air_temp_K",      # Numérica: Temperatura del aire
    "process_temp_K",  # Numérica: Temperatura del proceso
    "rpm",             # Numérica: Velocidad rotacional
    "torque_Nm",       # Numérica: Par de torsión
    "tool_wear_min"    # Numérica: Desgaste de la herramienta en minutos
]

# 5. Crear X (variables predictoras limpias) e y (etiqueta a predecir)
X = df[feature_cols].copy()
y = df[target].copy()

print(f"Variables predictoras (X): {X.columns.tolist()}")
print(f"Distribución del target (y):\n{y.value_counts(normalize=True) * 100}%")



# 1. División Train / Validation estratificada (mantiene la proporción de fallos del 3.4%)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# 2. Definir columnas numéricas (originales + las calculadas por add_features) y categóricas
categorical_cols = ["machine_type"]
numeric_cols = [
    "air_temp_K", "process_temp_K", "rpm", "torque_Nm", "tool_wear_min",
    "temp_diff_K", "power_W", "wear_torque"
]
# 3. Transformador para estandarizar numéricas y codificar categóricas
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(categories=[["L", "M", "H"]], handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols)
    ]
)
# 4. Pipeline completo: Features -> Transformaciones -> Modelo
# IMPORTANTE: class_weight='balanced' es clave por el desbalanceo del 3.4%
pipeline = Pipeline([
    ("features", FunctionTransformer(add_features)),
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42))
])
# 5. Entrenar el pipeline completo con los datos crudos
pipeline.fit(X_train, y_train)
# 6. Evaluar en el conjunto de validación
y_pred = pipeline.predict(X_val)
print("--- Informe de Clasificación ---")
print(classification_report(y_val, y_pred))

# Guardar el pipeline completo (.pkl)
joblib.dump(pipeline, '../models/cnc_failure_model.pkl')
print("¡Pipeline guardado con éxito en ../models/cnc_failure_model.pkl!")